In [0]:
dbutils.widgets.text(name="env", defaultValue="", label="Enter the envoronment in lower case")
db = dbutils.widgets.get("env")

In [0]:
%run ./commons

# Reading from bronze table


In [0]:
def read_BronzeTrafficTable(environment):
    print("Reading bronze table", end=' ')
    df_BronzeTraffic = spark.readStream.table(f"{environment}_catalog.bronze.`raw_traffic`")
    print(f'Reading {environment}_catalog.bronze.`raw_traffic` successful')
    return df_BronzeTraffic

# Getting count of electric vehicles by creating new column

In [0]:
def evcount(df):
    print("Calculating EV Counts", end=' ')
    from pyspark.sql.functions import col,sum
    df_ev = df.withColumn("Total_EV_Count", col("EV_Car")+ col("EV_Bike"))
    print(f'Calculating EV Counts successful')
    return df_ev

# Creating columns to get count of all motor vehicles

In [0]:
def Motor_count(df):
    print("Calculating All Motor Counts", end=' ')
    from pyspark.sql.functions import col,sum
    df_All_Motor = df.withColumn("Total_Motor_Vehicles", col("Pedal_cycles")+ col("Two_wheeled_motor_vehicles")+ col("Cars_and_taxis")+ col("Buses_and_coaches")+ col("LGV_Type")+ col("HGV_Type")+ col("EV_Car")+ col("EV_Bike"))
    print(f'Calculating All Motor Counts successful')
    return df_All_Motor


#creating transformed time column

In [0]:
def Create_TransformedTime(df):
    print("Creating Transformed Time", end=' ')
    from pyspark.sql.functions import current_timestamp
    df_transformedTime = df.withColumn("Transformed_Time", current_timestamp())
    print(f'Creating Transformed Time successful')
    return df_transformedTime


# Writing transformed data to silver traffic table

In [0]:
def Write_Traffic_Silver_Table(StreamingDF, environment):
    print("Writing Silver Table", end=' ')
    
    # Clear checkpoint if it exists to avoid conflicts
    checkpoint_path = checkpoint + "/SilverTrafficcloud/Checkpt/"
    dbutils.fs.rm(checkpoint_path, True)
    
    write_StreamSilver = (
        StreamingDF.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .queryName("SilverTrafficWriteStreamf")
        .trigger(availableNow=True)
        .toTable(f"{environment}_catalog.silver.Silver_Traffic")
    )
    
    write_StreamSilver.awaitTermination()
    print(f'Writing {environment}_catalog.silver.Silver_Traffic successful')

In [0]:
## Reading Bronze traffic data
df_trafficdata = read_BronzeTrafficTable(env)

## To remove duplicate rows
df_dups = remove_Dups(df_trafficdata)

## To replace any null value
Allcolumns = df_dups.schema.names
df_nulls = handle_NULLs(df_dups, Allcolumns)

## To get total ev count
df_ev = evcount(df_nulls)

## To get total motor count
df_motor = Motor_count(df_ev)

## Calling transformed time function
df_transformed = Create_TransformedTime(df_motor)

## Writing to Silver Table
Write_Traffic_Silver_Table(df_transformed, env)

Count

In [0]:
display(spark.sql(f"SELECT COUNT(*) FROM `{env}_catalog`.`silver`.`silver_traffic` Limit 10"))